In [1]:
import pandas as pd

In [31]:
pop = pd.read_csv("pop-municipios.csv")
selected_columns = [x for x in pop.columns if "unnamed:" not in x.lower()]
pop = pop[selected_columns].rename(
    columns = {'UF': 'uf',
 'COD. UF': 'codigo_uf',
 'COD. MUNIC': 'codigo_municipio',
 'NOME DO MUNICÍPIO': 'municipio',
 'POP. TOTAL': 'populacao'}
).drop(columns=['POP. COLETADA', 'POP. IMPUTADA'])

In [6]:
ava = pd.read_csv("avaliacoes_pntp_2024.csv", sep=";")

resp = pd.read_csv("respostas_pntp_2024.csv", sep=";")

/tmp/ipykernel_103731/4176094156.py:3: DtypeWarning: Columns (18) have mixed types. Specify dtype option on import or set low_memory=False.
  resp = pd.read_csv("respostas_pntp_2024.csv", sep=";")


In [34]:
# resp

In [32]:
pop

,uf,codigo_uf,codigo_municipio,municipio,populacao
0,RO,11,15,Alta Floresta D'Oeste,21 494
1,RO,11,23,Ariquemes,96 833
2,RO,11,31,Cabixi,5 351
3,RO,11,49,Cacoal,86 887
4,RO,11,56,Cerejeiras,15 890
...,...,...,...,...,...
5565,GO,52,22005,Vianópolis,14 956
5566,GO,52,22054,Vicentinópolis,8 768
5567,GO,52,22203,Vila Boa,4 215
5568,GO,52,22302,Vila Propício,5 815


# Selecionando a esfera municipal e quem tem link de avaliação

In [74]:
is_municipal = resp["esfera"] == "M"
has_link = resp["link_avaliacao"].notna()

is_prefeitura = municipais["poder"] == "E"

municipais = resp[is_municipal & has_link & is_prefeitura]


In [85]:
municipais.columns

Index(['ano_exercicio', 'questionario_id', 'entidade_id', 'entidade', 'ibge',
       'municipio', 'capital', 'uf', 'poder', 'esfera', 'dimensao_texto',
       'criterio_texto', 'exigibilidade', 'matriz', 'item_avaliacao',
       'resposta_avaliacao', 'link_avaliacao', 'observacao_avaliacao',
       'resposta_validacao', 'link_validacao', 'justificativa_validacao',
       'resposta_revisao', 'link_revisao', 'justificativa_revisao',
       'resposta_final'],
      dtype='object')

In [92]:
links = (
    municipais[["entidade_id", "entidade", "ibge", "uf", "municipio", "capital", "link_avaliacao"]]
    .drop_duplicates()
    .sort_values(["uf", "municipio"])
    .reset_index(drop=True)
)



In [190]:
from urllib.parse import urlparse
from tqdm import tqdm

avoid = pop["uf"].str.lower().drop_duplicates().to_list() + [
    "gov", "br", "go", "to", "pr", "com", "rn", "www", "pi", "site", "net",
    "instagram", "org", "facebook"
]

words = []

for url in tqdm(links["link_avaliacao"], total=len(links)):
    parts = urlparse(url)
    words.extend([x for x in parts.netloc.split(".") if (str(x).strip() != "") and (x not in avoid)])

words = pd.Series(words)

100%|███████████████████████████████| 160751/160751 [00:00<00:00, 181587.16it/s]


In [191]:
links[links["link_avaliacao"].str.contains('atende')].sample().iloc[0]["link_avaliacao"]

'https://altobelavista.atende.net/transparencia/item/consulta-estagiario'

In [196]:
words.value_counts().head(50)

transparencia            25628
acessoainformacao         8376
atende                    7717
cloud                     6201
betha                     5395
eloweb                    4067
portal                    3494
portaltp                  3019
cr2                       2783
equiplano                 2263
co                        1900
megasofttransparencia     1645
srv                       1450
govbr                     1402
ingadigital               1376
topsolutionsrn            1078
45                         985
177                        968
br:8079                    876
e-gov                      857
governotransparente        817
portaldatransparencia      790
leismunicipais             780
elotech                    767
oxy                        767
administracaopublica       743
asp                        721
app                        693
gp                         668
br:7474                    639
e-publica                  631
s2                         612
servicos

In [215]:
word = "atende.net"

teste = links[links["link_avaliacao"].str.contains(word)]

teste = (
    teste[["ibge", "uf", "municipio"]]
    .drop_duplicates().merge(pop[["uf", "municipio", "populacao"]], on=["uf", "municipio"], how="left")
    .sort_values("populacao", ascending=False)
    .reset_index(drop=True)
)

teste['populacao'] = teste['populacao'].apply(lambda x: int(str(x).replace(" ", "")))

print(f"QTE cidades = {len(teste)}")
print(f"Maior cidade = {teste.iloc[0]['municipio']}")
print(f"Pop. total = {teste['populacao'].sum()}")

QTE cidades = 147
Maior cidade = Campo Mourão
Pop. total = 5362691


In [210]:
# teste

In [187]:
word = "eloweb"

teste = links[links["link_avaliacao"].str.contains(word)]

teste = (
    teste[["ibge", "uf", "municipio"]]
    .drop_duplicates().merge(pop[["uf", "municipio", "populacao"]], on=["uf", "municipio"], how="left")
    .sort_values("populacao", ascending=False)
    .reset_index(drop=True)
)

teste['populacao'] = teste['populacao'].apply(lambda x: int(str(x).replace(" ", "")))

print(f"QTE cidades = {len(teste)}")
print(f"Maior cidade = {teste.iloc[0]['municipio']}")
print(f"Pop. total = {teste['populacao'].sum()}")

QTE cidades = 103
Maior cidade = Mariluz
Pop. total = 1819878


In [204]:
word = "cr2"

teste = links[links["link_avaliacao"].str.contains(word)]

teste = (
    teste[["ibge", "uf", "municipio"]]
    .drop_duplicates().merge(pop[["uf", "municipio", "populacao"]], on=["uf", "municipio"], how="left")
)

teste['populacao'] = teste['populacao'].apply(lambda x: int(str(x).replace(" ", "")))

teste = (
    teste
    .sort_values("populacao", ascending=False)
    .reset_index(drop=True)
)
print(f"QTE cidades = {len(teste)}")
print(f"Maior cidade = {teste.iloc[0]['municipio']}")
print(f"Pop. total = {teste['populacao'].sum()}")

QTE cidades = 142
Maior cidade = Castanhal
Pop. total = 4839342


In [203]:
word = "megasofttransparencia"

teste = links[links["link_avaliacao"].str.contains(word)]

teste = (
    teste[["ibge", "uf", "municipio"]]
    .drop_duplicates().merge(pop[["uf", "municipio", "populacao"]], on=["uf", "municipio"], how="left")
)

teste['populacao'] = teste['populacao'].apply(lambda x: int(str(x).replace(" ", "")))

teste = (
    teste
    .sort_values("populacao", ascending=False)
    .reset_index(drop=True)
)
print(f"QTE cidades = {len(teste)}")
print(f"Maior cidade = {teste.iloc[0]['municipio']}")
print(f"Pop. total = {teste['populacao'].sum()}")

QTE cidades = 56
Maior cidade = Uruaçu
Pop. total = 454303


In [202]:
word = "elotech"

teste = links[links["link_avaliacao"].str.contains(word)]

teste = (
    teste[["ibge", "uf", "municipio"]]
    .drop_duplicates().merge(pop[["uf", "municipio", "populacao"]], on=["uf", "municipio"], how="left")
)

teste['populacao'] = teste['populacao'].apply(lambda x: int(str(x).replace(" ", "")))

teste = (
    teste
    .sort_values("populacao", ascending=False)
    .reset_index(drop=True)
)
print(f"QTE cidades = {len(teste)}")
print(f"Maior cidade = {teste.iloc[0]['municipio']}")
print(f"Pop. total = {teste['populacao'].sum()}")

QTE cidades = 32
Maior cidade = Piraquara
Pop. total = 634573


In [201]:
word = "agilicloud"

teste = links[links["link_avaliacao"].str.contains(word)]

teste = (
    teste[["ibge", "uf", "municipio"]]
    .drop_duplicates().merge(pop[["uf", "municipio", "populacao"]], on=["uf", "municipio"], how="left")
)

teste['populacao'] = teste['populacao'].apply(lambda x: int(str(x).replace(" ", "")))

teste = (
    teste
    .sort_values("populacao", ascending=False)
    .reset_index(drop=True)
)
print(f"QTE cidades = {len(teste)}")
print(f"Maior cidade = {teste.iloc[0]['municipio']}")
print(f"Pop. total = {teste['populacao'].sum()}")

QTE cidades = 49
Maior cidade = Sorriso
Pop. total = 677304


In [199]:
word = "smarapd"

teste = links[links["link_avaliacao"].str.contains(word)]

teste = (
    teste[["ibge", "uf", "municipio"]]
    .drop_duplicates().merge(pop[["uf", "municipio", "populacao"]], on=["uf", "municipio"], how="left")
)

teste['populacao'] = teste['populacao'].apply(lambda x: int(str(x).replace(" ", "")))

teste = (
    teste
    .sort_values("populacao", ascending=False)
    .reset_index(drop=True)
)

print(f"QTE cidades = {len(teste)}")
print(f"Maior cidade = {teste.iloc[0]['municipio']}")
print(f"Pop. total = {teste['populacao'].sum()}")

QTE cidades = 17
Maior cidade = Bauru
Pop. total = 1811305


In [207]:
word = "topsolutionsrn"

teste = links[links["link_avaliacao"].str.contains(word)]

teste = (
    teste[["ibge", "uf", "municipio"]]
    .drop_duplicates().merge(pop[["uf", "municipio", "populacao"]], on=["uf", "municipio"], how="left")
)

teste['populacao'] = teste['populacao'].fillna("0").apply(lambda x: int(str(x).replace(" ", "")))

teste = (
    teste
    .sort_values("populacao", ascending=False)
    .reset_index(drop=True)
)

print(f"QTE cidades = {len(teste)}")
print(f"Maior cidade = {teste.iloc[0]['municipio']}")
print(f"Pop. total = {teste['populacao'].sum()}")

QTE cidades = 53
Maior cidade = Parnamirim
Pop. total = 1004195


In [216]:
word = "ingadigital"

teste = links[links["link_avaliacao"].str.contains(word)]

teste = (
    teste[["ibge", "uf", "municipio"]]
    .drop_duplicates().merge(pop[["uf", "municipio", "populacao"]], on=["uf", "municipio"], how="left")
)

teste['populacao'] = teste['populacao'].fillna("0").apply(lambda x: int(str(x).replace(" ", "")))

teste = (
    teste
    .sort_values("populacao", ascending=False)
    .reset_index(drop=True)
)

print(f"QTE cidades = {len(teste)}")
print(f"Maior cidade = {teste.iloc[0]['municipio']}")
print(f"Pop. total = {teste['populacao'].sum()}")

QTE cidades = 77
Maior cidade = Marialva
Pop. total = 737337


In [217]:
word = "oxy"

teste = links[links["link_avaliacao"].str.contains(word)]

teste = (
    teste[["ibge", "uf", "municipio"]]
    .drop_duplicates().merge(pop[["uf", "municipio", "populacao"]], on=["uf", "municipio"], how="left")
)

teste['populacao'] = teste['populacao'].fillna("0").apply(lambda x: int(str(x).replace(" ", "")))

teste = (
    teste
    .sort_values("populacao", ascending=False)
    .reset_index(drop=True)
)

print(f"QTE cidades = {len(teste)}")
print(f"Maior cidade = {teste.iloc[0]['municipio']}")
print(f"Pop. total = {teste['populacao'].sum()}")

QTE cidades = 32
Maior cidade = Piraquara
Pop. total = 634573


In [140]:


avoid = pop["uf"].str.lower().drop_duplicates().to_list() + [
    "gov", "br", "go", "to", "pr", "com", "rn", "www", "pi", "site",
]

url = municipais.sample().iloc[0]["link_avaliacao"]

print(url)

parts = urlparse(url)

[x for x in parts.netloc.split(".") if x not in avoid]

https://transparencia.e-publica.net/epublica-portal/#/salgado_filho/portal/publicacaoarquivoGroupFile?params=%7B%22parent%22:%22162%22,%22property%22:%22publicacaoArquivo.nivel02Id%22%7D&entidade=1524


['transparencia', 'e-publica', 'net']

# ERPs
- https://cr2.co/
- https://www12.senado.leg.br/interlegis

- https://transparencia.agilicloud.com.br/
- https://fiorilli.com.br/
- https://www.centi.com.br/portal/: (https://acessoainformacao.itaberai.go.gov.br/ - API)
- https://www.topsolutionscloud.com/br/ (https://pmjardimseridorn.transparencia.topsolutionsrn.com.br/servidores - API)

['ro',
 'ac',
 'am',
 'rr',
 'pa',
 'ap',
 'to',
 'ma',
 'pi',
 'ce',
 'rn',
 'pb',
 'pe',
 'al',
 'se',
 'ba',
 'mg',
 'es',
 'rj',
 'sp',
 'pr',
 'sc',
 'rs',
 'ms',
 'mt',
 'go',
 'df']